In [1]:
import os
import random
import pickle
import shutil
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scipy.io
from scipy.io import loadmat

import mne
import tensorflow as tf

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from tensorflow.keras.constraints import max_norm
from tensorflow.keras.layers import (
    Activation,
    AveragePooling2D,
    BatchNormalization,
    Conv2D,
    Dense,
    DepthwiseConv2D,
    Dropout,
    Flatten,
    Input,
    Reshape,
    SeparableConv2D,
    SpatialDropout2D,
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import os
from typing import Tuple

import tensorflow as tf
from tensorflow.keras import Input, Model, layers, models

import numpy as np
from scipy.signal import welch, butter, filtfilt
import os
import gc
import random
import numpy as np
import pandas as pd
import tensorflow as tf


2026-04-29 17:29:47.559509: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777483787.902726      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777483788.010834      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777483788.880240      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777483788.880282      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777483788.880285      22 computation_placer.cc:177] computation placer alr

In [2]:
class EEGDataset_ADHD_TF:
    """
    Subject-level dataset (one .mat file = one subject)
    Each item: (fname, eeg_epochs_np, label_int)
      eeg_epochs_np: (E, C, T)
    """

    def __init__(self, adhd_dir, control_dir,
                 lowcut=0.5, highcut=60.0, notch=50.0,
                 window=2.0, overlap=0.5,
                 default_fs=128):

        self.samples = []
        self.lowcut = float(lowcut)
        self.highcut = float(highcut)
        self.notch = float(notch)
        self.window = float(window)
        self.overlap = float(overlap)
        self.default_fs = int(default_fs)

        self._process_folder(adhd_dir, label=1)
        self._process_folder(control_dir, label=0)

        if len(self.samples) == 0:
            raise ValueError("No subjects loaded. Check folders and file contents.")

    def _process_folder(self, folder, label):
        if not os.path.isdir(folder):
            raise ValueError(f"Directory does not exist: {folder}")

        for fname in sorted(os.listdir(folder)):
            if not fname.lower().endswith(".mat"):
                continue
            mat_path = os.path.join(folder, fname)
            eeg = self._process_mat(mat_path)
            if eeg is not None:
                self.samples.append((fname, eeg.astype(np.float32), int(label)))

    def _process_mat(self, file_path):
        mat = loadmat(file_path)

        # Variable name is usually the same as filename (e.g., v10p.mat -> "v10p")
        key = os.path.splitext(os.path.basename(file_path))[0]
        if key not in mat:
            # fallback: first 2D array
            key = None
            for k, v in mat.items():
                if isinstance(v, np.ndarray) and v.ndim == 2 and v.size > 0:
                    key = k
                    break
            if key is None:
                return None

        data = np.asarray(mat[key], dtype=np.float64)  # often (T, C)

        # Find fs if present
        fs = None
        for k in mat.keys():
            kl = k.lower()
            if ("fs" in kl) or ("freq" in kl) or ("sampling" in kl) or ("sfreq" in kl):
                try:
                    fs = int(np.squeeze(mat[k]))
                    break
                except Exception:
                    fs = None
        if fs is None or fs <= 0:
            fs = self.default_fs

        # Ensure (C, T) for MNE
        if data.ndim != 2:
            return None

        if data.shape[1] <= 256 and data.shape[0] > data.shape[1]:
            data = data.T  # (C, T)

        n_ch, n_times = data.shape
        min_samples = int(np.ceil(self.window * fs))
        if n_times < min_samples:
            return None

        ch_names = [f"Ch{i+1}" for i in range(n_ch)]
        info = mne.create_info(ch_names=ch_names, sfreq=fs, ch_types=["eeg"] * n_ch)
        raw = mne.io.RawArray(data, info, verbose=False)

        raw.set_eeg_reference("average", verbose=False)
        raw.notch_filter(freqs=[self.notch], method="iir", verbose=False)
        raw.filter(self.lowcut, self.highcut, method="iir", verbose=False)

        step = self.window * (1.0 - self.overlap)
        if step <= 0:
            raise ValueError("overlap must be < 1.0 so that step > 0")

        epochs = mne.make_fixed_length_epochs(
            raw,
            duration=self.window,
            overlap=self.window - step,
            preload=True,
            verbose=False
        )

        eeg_data = epochs.get_data()  # (E, C, T)
        if eeg_data.size == 0:
            return None

        return eeg_data

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [3]:
# Build epoch-level arrays from subject-level dataset
def build_epoch_arrays(dataset_obj):
    X_list, y_list, g_list = [], [], []

    for i in range(len(dataset_obj)):
        name, eeg_epochs, label = dataset_obj[i]  # (E,C,T)
        E = int(eeg_epochs.shape[0])

        X_list.append(eeg_epochs)
        y_list.append(np.full((E,), int(label), dtype=np.int32))
        g_list.append(np.full((E,), str(name), dtype=object))

    X = np.concatenate(X_list, axis=0).astype(np.float32)   # (N,C,T)
    y = np.concatenate(y_list, axis=0).astype(np.int32)     # (N,)
    groups = np.concatenate(g_list, axis=0)                 # (N,) subject-name per epoch

    return X, y, groups


# Subject-level labels derived from epoch-level groups/y
def build_subject_table_from_epochs(groups_epoch, y_epoch):
    subj_names = np.unique(groups_epoch.astype(str))
    subj_labels = []

    for s in subj_names:
        ys = np.unique(y_epoch[groups_epoch.astype(str) == s])
        if len(ys) != 1:
            raise ValueError(f"Subject {s} has multiple labels across epochs: {ys}")
        subj_labels.append(int(ys[0]))

    return subj_names, np.array(subj_labels, dtype=np.int32)


def epoch_indices_from_subjects(groups_epoch, subject_names_subset):
    subject_names_subset = np.array(subject_names_subset).astype(str)
    return np.where(np.isin(groups_epoch.astype(str), subject_names_subset))[0]


# tf.data builders
def make_ds_from_indices(X, y, groups, idxs, training, with_groups, batch_size, seed):
    x = X[idxs]
    yy = y[idxs]

    if with_groups:
        gg = groups[idxs].astype(str)
        ds = tf.data.Dataset.from_tensor_slices((x, yy, gg))
    else:
        ds = tf.data.Dataset.from_tensor_slices((x, yy))

    if training:
        ds = ds.shuffle(len(idxs), seed=seed, reshuffle_each_iteration=True)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# Validation Balanced Accuracy callback (epoch-level)
class ValBalancedAccuracy(tf.keras.callbacks.Callback):
    def __init__(self, val_ds_xy):
        super().__init__()
        self.val_ds = val_ds_xy
        self.best = -np.inf
        self.last = None

    def on_epoch_end(self, epoch, logs=None):
        y_true, y_pred = [], []
        for xb, yb in self.val_ds:
            prob = self.model(xb, training=False).numpy().reshape(-1)
            pred = (prob >= 0.5).astype(int)
            y_true.extend(yb.numpy().tolist())
            y_pred.extend(pred.tolist())

        bacc = balanced_accuracy_score(y_true, y_pred)
        self.last = float(bacc)
        self.best = max(self.best, self.last)
        print(f" - val_balanced_accuracy: {self.last:.4f}", end="")

# Plotting functions
def plot_fold_training_curve(history, fold_id, save_path, metrics=("loss", "acc", "auc")):
    n_metrics = len(metrics)
    fig, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 4))
    if n_metrics == 1:
        axes = [axes]
    fig.suptitle(f"Fold {fold_id + 1} — Training Curves (Final Retrain)", fontsize=13)
    epochs_range = range(1, len(history["loss"]) + 1)
    for col, m in enumerate(metrics):
        ax = axes[col]
        if m in history:
            ax.plot(epochs_range, history[m], label=f"train_{m}", linewidth=1.2)
        val_key = f"val_{m}"
        if val_key in history:
            ax.plot(epochs_range, history[val_key], label=val_key, linewidth=1.2, linestyle="--")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(m)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {save_path}")

# Fold-level training curves for all folds in a grid + overlay of all folds per metric
def plot_all_training_curves(fold_histories, save_dir, metrics=("loss", "acc", "auc")):
    n_folds = len(fold_histories)
    n_metrics = len(metrics)

    fig, axes = plt.subplots(n_folds, n_metrics, figsize=(5 * n_metrics, 3.5 * n_folds), squeeze=False)
    fig.suptitle("Training Curves per Fold (Final Retrain)", fontsize=14, y=1.01)
    for row, fid in enumerate(sorted(fold_histories.keys())):
        hist = fold_histories[fid]
        epochs_range = range(1, len(hist["loss"]) + 1)
        for col, m in enumerate(metrics):
            ax = axes[row, col]
            if m in hist:
                ax.plot(epochs_range, hist[m], label=f"train_{m}", linewidth=1.0)
            val_key = f"val_{m}"
            if val_key in hist:
                ax.plot(epochs_range, hist[val_key], label=val_key, linewidth=1.0, linestyle="--")
            ax.set_title(f"Fold {fid + 1} — {m}", fontsize=9)
            ax.set_xlabel("Epoch")
            ax.set_ylabel(m)
            ax.legend(fontsize=6)
            ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, "training_curves_all_folds.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")

    fig2, axes2 = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 4))
    if n_metrics == 1:
        axes2 = [axes2]
    fig2.suptitle("Training Curves Overlay (All Folds)", fontsize=14)
    for col, m in enumerate(metrics):
        ax = axes2[col]
        for fid in sorted(fold_histories.keys()):
            hist = fold_histories[fid]
            epochs_range = range(1, len(hist["loss"]) + 1)
            val_key = f"val_{m}"
            if val_key in hist:
                ax.plot(epochs_range, hist[val_key], label=f"Fold {fid + 1}", linewidth=1.0, alpha=0.7)
            elif m in hist:
                ax.plot(epochs_range, hist[m], label=f"Fold {fid + 1}", linewidth=1.0, alpha=0.7)
        ax.set_title(f"All Folds — {m}", fontsize=11)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(m)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path2 = os.path.join(save_dir, "training_curves_overlay.png")
    fig2.savefig(path2, dpi=150, bbox_inches="tight")
    plt.close(fig2)
    print(f"Saved: {path2}")

# Fold-level confusion matrix + accumulated confusion matrix across all folds (subject-level)
def plot_fold_confusion_matrix(cm, fold_id, save_path, class_names=("Control", "ADHD")):
    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"Fold {fold_id + 1} — Confusion Matrix (Subject-Level)", fontsize=11)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {save_path}")

# Fold-level confusion matrix + accumulated confusion matrix across all folds (subject-level)
def plot_all_confusion_matrices(fold_cms, super_cm, save_dir, class_names=("Control", "ADHD")):
    n_folds = len(fold_cms)
    ncols = min(n_folds, 5)
    nrows = int(np.ceil(n_folds / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), squeeze=False)
    fig.suptitle("Confusion Matrices per Fold (Subject-Level)", fontsize=14, y=1.01)
    for i, fid in enumerate(sorted(fold_cms.keys())):
        row, col = divmod(i, ncols)
        ax = axes[row][col]
        disp = ConfusionMatrixDisplay(fold_cms[fid], display_labels=class_names)
        disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
        ax.set_title(f"Fold {fid + 1}", fontsize=10)
    for j in range(i + 1, nrows * ncols):
        row, col = divmod(j, ncols)
        axes[row][col].axis("off")
    plt.tight_layout()
    path = os.path.join(save_dir, "confusion_matrices_all_folds.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")

    fig2, ax2 = plt.subplots(1, 1, figsize=(5, 5))
    disp2 = ConfusionMatrixDisplay(super_cm, display_labels=class_names)
    disp2.plot(ax=ax2, cmap="Oranges", colorbar=False, values_format="d")
    ax2.set_title("Accumulated Confusion Matrix (All Folds)", fontsize=12)
    plt.tight_layout()
    path2 = os.path.join(save_dir, "confusion_matrix_accumulated.png")
    fig2.savefig(path2, dpi=150, bbox_inches="tight")
    plt.close(fig2)
    print(f"Saved: {path2}")

In [4]:
# Build epoch-level arrays from subject-level dataset
def build_epoch_arrays(dataset_obj):
    X_list, y_list, g_list = [], [], []

    for i in range(len(dataset_obj)):
        name, eeg_epochs, label = dataset_obj[i]  # (E,C,T)
        E = int(eeg_epochs.shape[0])

        X_list.append(eeg_epochs)
        y_list.append(np.full((E,), int(label), dtype=np.int32))
        g_list.append(np.full((E,), str(name), dtype=object))

    X = np.concatenate(X_list, axis=0).astype(np.float32)   # (N,C,T)
    y = np.concatenate(y_list, axis=0).astype(np.int32)     # (N,)
    groups = np.concatenate(g_list, axis=0)                 # (N,) subject-name per epoch

    return X, y, groups


# Subject-level labels derived from epoch-level groups/y
def build_subject_table_from_epochs(groups_epoch, y_epoch):
    subj_names = np.unique(groups_epoch.astype(str))
    subj_labels = []

    for s in subj_names:
        ys = np.unique(y_epoch[groups_epoch.astype(str) == s])
        if len(ys) != 1:
            raise ValueError(f"Subject {s} has multiple labels across epochs: {ys}")
        subj_labels.append(int(ys[0]))

    return subj_names, np.array(subj_labels, dtype=np.int32)


def epoch_indices_from_subjects(groups_epoch, subject_names_subset):
    subject_names_subset = np.array(subject_names_subset).astype(str)
    return np.where(np.isin(groups_epoch.astype(str), subject_names_subset))[0]


# tf.data builders
def make_ds_from_indices(X, y, groups, idxs, training, with_groups, batch_size, seed):
    x = X[idxs]
    yy = y[idxs]

    if with_groups:
        gg = groups[idxs].astype(str)
        ds = tf.data.Dataset.from_tensor_slices((x, yy, gg))
    else:
        ds = tf.data.Dataset.from_tensor_slices((x, yy))

    if training:
        ds = ds.shuffle(len(idxs), seed=seed, reshuffle_each_iteration=True)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

# Validation Balanced Accuracy callback (epoch-level)
class ValBalancedAccuracy(tf.keras.callbacks.Callback):
    def __init__(self, val_ds_xy):
        super().__init__()
        self.val_ds = val_ds_xy
        self.best = -np.inf
        self.last = None

    def on_epoch_end(self, epoch, logs=None):
        y_true, y_pred = [], []
        for xb, yb in self.val_ds:
            prob = self.model(xb, training=False).numpy().reshape(-1)
            pred = (prob >= 0.5).astype(int)
            y_true.extend(yb.numpy().tolist())
            y_pred.extend(pred.tolist())

        bacc = balanced_accuracy_score(y_true, y_pred)
        self.last = float(bacc)
        self.best = max(self.best, self.last)
        print(f" - val_balanced_accuracy: {self.last:.4f}", end="")

# Plotting functions
def plot_fold_training_curve(history, fold_id, save_path, metrics=("loss", "acc", "auc")):
    n_metrics = len(metrics)
    fig, axes = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 4))
    if n_metrics == 1:
        axes = [axes]
    fig.suptitle(f"Fold {fold_id + 1} — Training Curves (Final Retrain)", fontsize=13)
    epochs_range = range(1, len(history["loss"]) + 1)
    for col, m in enumerate(metrics):
        ax = axes[col]
        if m in history:
            ax.plot(epochs_range, history[m], label=f"train_{m}", linewidth=1.2)
        val_key = f"val_{m}"
        if val_key in history:
            ax.plot(epochs_range, history[val_key], label=val_key, linewidth=1.2, linestyle="--")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(m)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {save_path}")

# Fold-level training curves for all folds in a grid + overlay of all folds per metric
def plot_all_training_curves(fold_histories, save_dir, metrics=("loss", "acc", "auc")):
    n_folds = len(fold_histories)
    n_metrics = len(metrics)

    fig, axes = plt.subplots(n_folds, n_metrics, figsize=(5 * n_metrics, 3.5 * n_folds), squeeze=False)
    fig.suptitle("Training Curves per Fold (Final Retrain)", fontsize=14, y=1.01)
    for row, fid in enumerate(sorted(fold_histories.keys())):
        hist = fold_histories[fid]
        epochs_range = range(1, len(hist["loss"]) + 1)
        for col, m in enumerate(metrics):
            ax = axes[row, col]
            if m in hist:
                ax.plot(epochs_range, hist[m], label=f"train_{m}", linewidth=1.0)
            val_key = f"val_{m}"
            if val_key in hist:
                ax.plot(epochs_range, hist[val_key], label=val_key, linewidth=1.0, linestyle="--")
            ax.set_title(f"Fold {fid + 1} — {m}", fontsize=9)
            ax.set_xlabel("Epoch")
            ax.set_ylabel(m)
            ax.legend(fontsize=6)
            ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path = os.path.join(save_dir, "training_curves_all_folds.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")

    fig2, axes2 = plt.subplots(1, n_metrics, figsize=(5 * n_metrics, 4))
    if n_metrics == 1:
        axes2 = [axes2]
    fig2.suptitle("Training Curves Overlay (All Folds)", fontsize=14)
    for col, m in enumerate(metrics):
        ax = axes2[col]
        for fid in sorted(fold_histories.keys()):
            hist = fold_histories[fid]
            epochs_range = range(1, len(hist["loss"]) + 1)
            val_key = f"val_{m}"
            if val_key in hist:
                ax.plot(epochs_range, hist[val_key], label=f"Fold {fid + 1}", linewidth=1.0, alpha=0.7)
            elif m in hist:
                ax.plot(epochs_range, hist[m], label=f"Fold {fid + 1}", linewidth=1.0, alpha=0.7)
        ax.set_title(f"All Folds — {m}", fontsize=11)
        ax.set_xlabel("Epoch")
        ax.set_ylabel(m)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    path2 = os.path.join(save_dir, "training_curves_overlay.png")
    fig2.savefig(path2, dpi=150, bbox_inches="tight")
    plt.close(fig2)
    print(f"Saved: {path2}")

# Fold-level confusion matrix + accumulated confusion matrix across all folds (subject-level)
def plot_fold_confusion_matrix(cm, fold_id, save_path, class_names=("Control", "ADHD")):
    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"Fold {fold_id + 1} — Confusion Matrix (Subject-Level)", fontsize=11)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {save_path}")

# Fold-level confusion matrix + accumulated confusion matrix across all folds (subject-level)
def plot_all_confusion_matrices(fold_cms, super_cm, save_dir, class_names=("Control", "ADHD")):
    n_folds = len(fold_cms)
    ncols = min(n_folds, 5)
    nrows = int(np.ceil(n_folds / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), squeeze=False)
    fig.suptitle("Confusion Matrices per Fold (Subject-Level)", fontsize=14, y=1.01)
    for i, fid in enumerate(sorted(fold_cms.keys())):
        row, col = divmod(i, ncols)
        ax = axes[row][col]
        disp = ConfusionMatrixDisplay(fold_cms[fid], display_labels=class_names)
        disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
        ax.set_title(f"Fold {fid + 1}", fontsize=10)
    for j in range(i + 1, nrows * ncols):
        row, col = divmod(j, ncols)
        axes[row][col].axis("off")
    plt.tight_layout()
    path = os.path.join(save_dir, "confusion_matrices_all_folds.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")

    fig2, ax2 = plt.subplots(1, 1, figsize=(5, 5))
    disp2 = ConfusionMatrixDisplay(super_cm, display_labels=class_names)
    disp2.plot(ax=ax2, cmap="Oranges", colorbar=False, values_format="d")
    ax2.set_title("Accumulated Confusion Matrix (All Folds)", fontsize=12)
    plt.tight_layout()
    path2 = os.path.join(save_dir, "confusion_matrix_accumulated.png")
    fig2.savefig(path2, dpi=150, bbox_inches="tight")
    plt.close(fig2)
    print(f"Saved: {path2}")

In [5]:
def segmentar_senales(db, labels):
    """
    Divide las señales EEG en segmentos de 512 instantes con un traslape del 50%.

    Args:
        db (dict): Diccionario donde las claves son los nombres de los sujetos y los valores
                   son matrices de forma CxT_i (C = canales, T_i = tiempo).

    Returns:
        tuple:
            - segmentos: array de segmentos
            - y: array de etiquetas
            - sbjs: lista de sujetos por segmento
            - window_ids: lista con identificador de ventana por segmento
    """
    segmento_tamano = 512
    paso = int(segmento_tamano * 0.5)  # 50% overlap
    i = 0

    segmentos = []
    y = []
    sbjs = []
    window_ids = []

    for sujeto, senal in db.items():
        C, T = senal.shape
        window_count = 1

        for inicio in range(0, T - segmento_tamano + 1, paso):
            segmento = senal[:, inicio:inicio + segmento_tamano]
            segmentos.append(segmento)
            y.append(labels[i])
            sbjs.append(sujeto)
            window_ids.append(f"Window {window_count}")
            window_count += 1

        i += 1

    return np.array(segmentos), np.array(y), sbjs, window_ids


In [6]:
# ---------------------------------------------------------------------------
# 1. Positional encoding (fixed sinusoidal, Vaswani et al. 2017)
# ---------------------------------------------------------------------------

class PositionalEncoding(layers.Layer):
    """Adds 1-D sinusoidal positional embeddings to a sequence tensor."""

    def __init__(self, d_model: int, max_len: int = 5000, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model

        # positions × channels → (max_len, d_model)
        pos = tf.cast(tf.range(max_len)[:, tf.newaxis], tf.float32)  # (L,1)
        i = tf.cast(tf.range(d_model)[tf.newaxis, :], tf.float32)    # (1,D)

        exponent = (2.0 * (i // 2)) / tf.cast(d_model, tf.float32)   # float32
        angle_rates = tf.constant(1.0, tf.float32) / tf.pow(10000.0, exponent)
        angle_rads = pos * angle_rates                               # (L,D)

        # Even indices → sin, odd indices → cos
        sin = tf.sin(angle_rads[:, 0::2])
        cos = tf.cos(angle_rads[:, 1::2])
        pe = tf.reshape(tf.concat([sin, cos], axis=-1), (max_len, d_model))
        self.pos_encoding = pe[tf.newaxis]                           # (1,L,D)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        return tf.cast(x, tf.float32) + self.pos_encoding[:, :seq_len, :]


# ---------------------------------------------------------------------------
# 2. Transformer encoder block (MHA + FFN)
# ---------------------------------------------------------------------------

class TransformerEncoderBlock(layers.Layer):
    def __init__(self, d_model: int = 64, num_heads: int = 4, d_ff: int = 128,
                 rate: float = 0.1, **kwargs):
        super().__init__(**kwargs)
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.ffn = models.Sequential([
            layers.Dense(d_ff, activation="relu"),
            layers.Dense(d_model),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, x, training=False):
        attn_output = self.attn(x, x, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)


# ---------------------------------------------------------------------------
# 3. Stream builder (spectral / temporal / spatial)
# ---------------------------------------------------------------------------

def build_stream(name: str, seq_len: int, feat_dim: int, d_model: int = 64,
                 num_layers: int = 2) -> Tuple[Input, layers.Layer]:
    """Create one transformer stream."""
    inp = Input(shape=(seq_len, feat_dim), name=f"{name}_input")
    x = layers.Dense(d_model)(inp)              # linear projection
    x = PositionalEncoding(d_model)(x)          # add PE
    for _ in range(num_layers):
        x = TransformerEncoderBlock(d_model=d_model)(x)
    x = layers.GlobalAveragePooling1D()(x)      # sequence → vector
    x = layers.Dense(128, activation="relu")(x)  # per-stream decoder
    return inp, x


# ---------------------------------------------------------------------------
# 4. Full multi-stream model
# ---------------------------------------------------------------------------

def build_eeg_attention_model(
    freq_shape: Tuple[int, int],
    temp_shape: Tuple[int, int],
    spat_shape: Tuple[int, int],
    d_model: int = 64,
    num_layers: int = 2,
) -> Model:
    """Assemble the three-stream transformer network."""
    freq_in, freq_vec = build_stream("freq", *freq_shape, d_model=d_model, num_layers=num_layers)
    temp_in, temp_vec = build_stream("temp", *temp_shape, d_model=d_model, num_layers=num_layers)
    spat_in, spat_vec = build_stream("spat", *spat_shape, d_model=d_model, num_layers=num_layers)

    concat = layers.concatenate([freq_vec, temp_vec, spat_vec], name="concat_streams")
    x = layers.Dense(128, activation="relu")(concat)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(2, activation="softmax", name="attention_state")(x)

    return Model(inputs=[freq_in, temp_in, spat_in], outputs=output, name="EEG_Attention_Transformer")


def band_power_psd(x, fs, bands):
    """Compute average power in each frequency band for one trial."""
    freqs, psd = welch(x, fs=fs, nperseg=fs*2)          # (freq,)
    pow_in_band = []
    for low, high in bands:
        mask = (freqs >= low) & (freqs < high)
        pow_in_band.append(psd[..., mask].mean())
    return np.array(pow_in_band)                         # (n_bands,)

import numpy as np
from scipy.signal import welch

def prepare_streams_4s(X, fs=128, n_win=10):
    """
    X : ndarray (N, C, 512)   4‑second segments @128 Hz
    Returns freq, temp, spat ready for model.fit
    """
    N, C, T = X.shape
    assert T == 512, "Expecting exactly 512‑sample segments (4 s @128 Hz)"

    # ── 1. Spectral stream (20 log bands) ────────────────────────
    log_bands = np.logspace(np.log10(1), np.log10(fs / 2), 21)
    bands = list(zip(log_bands[:-1], log_bands[1:]))

    def band_power(signal):
        f, Pxx = welch(signal, fs=fs, nperseg=512)     # 512‑pt FFT → all bands covered
        return np.array([
            Pxx[(f >= lo) & (f < hi)].mean()
            if np.any((f >= lo) & (f < hi)) else 0.0   # avoid empty‑slice warning
            for lo, hi in bands
        ])

    freq_feats = np.stack([[band_power(ch) for ch in trial] for trial in X])
    freq = freq_feats.mean(axis=1)[..., None]          # (N, 20, 1)

    # ── 2. Temporal stream (10 × 51 samples = 510 samples) ──────
    T_win = T // n_win           # 512 // 10 = 51
    usable = n_win * T_win       # 510
    x_trim = X[:, :, :usable]    # drop the last 2 samples
    windows = x_trim.reshape(N, C, n_win, T_win)   # (N, C, 10, 51)

    # simple mean amplitude; swap in richer stats if you like
    temp = windows.mean(axis=(1, 3))[..., None]        # (N, 10, 1)

    # ── 3. Spatial stream (per‑channel RMS) ─────────────────────
    spat = np.sqrt((X ** 2).mean(axis=2))[..., None]   # (N, C, 1)

    return (freq.astype("float32"),
            temp.astype("float32"),
            spat.astype("float32"))

In [7]:
# =========================================================
# 1) RUTAS BASE DE DATOS
# =========================================================

root = "/kaggle/input/datasets/javierceron/tdha-data"
adhd_dir = os.path.join(root, "ADHD")
control_dir = os.path.join(root, "Control")

print("ADHD dir existe:", os.path.isdir(adhd_dir))
print("Control dir existe:", os.path.isdir(control_dir))

# =========================================================
# 2) SUJETOS A EXCLUIR
# =========================================================

EXCLUDED_SUBJECTS = {"v56p"}   

# =========================================================
# 3) CREAR DATASET CON TU RUTINA EEGDataset_ADHD_TF
# =========================================================

dataset_tf = EEGDataset_ADHD_TF(
    adhd_dir=adhd_dir,
    control_dir=control_dir,
    lowcut=0.5,
    highcut=60.0,
    notch=50.0,
    window=2.0,
    overlap=0.5,
    default_fs=128
)

# =========================================================
# 4) EXCLUIR SUJETOS PROBLEMÁTICOS
# =========================================================

n_before = len(dataset_tf.samples)

dataset_tf.samples = [
    sample for sample in dataset_tf.samples
    if os.path.splitext(sample[0])[0] not in EXCLUDED_SUBJECTS
]

n_after = len(dataset_tf.samples)
removed = n_before - n_after

print(f"Excluded subjects: {sorted(EXCLUDED_SUBJECTS)}")
print(f"Subjects removed: {removed}")
print(f"Remaining subjects: {n_after}")

if len(dataset_tf.samples) == 0:
    raise ValueError("No subjects left after exclusion. Check EXCLUDED_SUBJECTS.")

# =========================================================
# 5) CONSTRUIR ARREGLOS EPOCH-LEVEL
# =========================================================

X, y, groups = build_epoch_arrays(dataset_tf)

print("\nEpoch arrays:")
print("  X:", X.shape, X.dtype)
print("  y:", y.shape, "labels:", np.unique(y))
print("  groups:", groups.shape, "n_subjects:", len(np.unique(groups.astype(str))))

N_CH, N_T = int(X.shape[1]), int(X.shape[2])
print("\nModel input shape (C,T):", (N_CH, N_T))

subj_names, subj_labels = build_subject_table_from_epochs(groups, y)

print("\nSubject table:")
print("  n_subjects:", len(subj_names))
print("  Control:", int(np.sum(subj_labels == 0)), "| ADHD:", int(np.sum(subj_labels == 1)))

model = build_eeg_attention_model(
    freq_shape=(20, 1),
    temp_shape=(10, 1),
    spat_shape=(N_CH, 1),
    d_model=64,
    num_layers=2,
)

print("\n================ MODEL SUMMARY ================\n")

model.summary(
    line_length=150,
    expand_nested=True
)

ADHD dir existe: True
Control dir existe: True
Excluded subjects: ['v56p']
Subjects removed: 1
Remaining subjects: 120

Epoch arrays:
  X: (16640, 19, 256) float32
  y: (16640,) labels: [0 1]
  groups: (16640,) n_subjects: 120

Model input shape (C,T): (19, 256)

Subject table:
  n_subjects: 120
  Control: 59 | ADHD: 61


I0000 00:00:1777483839.066445      22 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0



================ MODEL SUMMARY ================



Model: "EEG_Attention_Transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━
┃ Layer (type)                               ┃ Output Shape                         ┃                 Param # ┃ Con
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━
│ freq_input (InputLayer)                    │ (None, 20, 1)                        │                       0 │ -  
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ temp_input (InputLayer)                    │ (None, 10, 1)                        │                       0 │ -  
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ spat_input (InputLayer)                    │ (None, 19, 1)                        │                       0 │ -  
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ dense (Dense)                              │ (None, 20, 64)                       │                     128 │ fre
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ dense_6 (Dense)                            │ (None, 10, 64)                       │                     128 │ tem
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ dense_12 (Dense)                           │ (None, 19, 64)                       │                     128 │ spa
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ positional_encoding (PositionalEncoding)   │ (None, 20, 64)                       │                       0 │ den
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ positional_encoding_1 (PositionalEncoding) │ (None, 10, 64)                       │                       0 │ den
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ positional_encoding_2 (PositionalEncoding) │ (None, 19, 64)                       │                       0 │ den
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ transformer_encoder_block                  │ (None, 20, 64)                       │                  83,200 │ pos
│ (TransformerEncoderBlock)                  │                                      │                         │    
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ transformer_encoder_block_2                │ (None, 10, 64)                       │                  83,200 │ pos
│ (TransformerEncoderBlock)                  │                                      │                         │    
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ transformer_encoder_block_4                │ (None, 19, 64)                       │                  83,200 │ pos
│ (TransformerEncoderBlock)                  │                                      │                         │    
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ transformer_encoder_block_1                │ (None, 20, 64)                       │                  83,200 │ tra
│ (TransformerEncoderBlock)                  │                                      │                         │    
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ transformer_encoder_block_3                │ (None, 10, 64)                       │                  83,200 │ tra
│ (TransformerEncoderBlock)                  │          

 Total params: 574,082 (2.19 MB)

 Trainable params: 574,082 (2.19 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
# SEED = 123
# MODEL_NAME = "eeg_attention_transformer"

# N_FOLDS = 5
# BATCH_SIZE = 64
# EPOCHS_FINAL = 100

# RESULTS_ROOT = "/kaggle/working/results_eeg_attention_transformer_fixed_seed123"
# CURVES_DIR   = os.path.join(RESULTS_ROOT, "training_curves")
# MODELS_DIR   = os.path.join(RESULTS_ROOT, "models")
# CM_DIR       = os.path.join(RESULTS_ROOT, "cm")
# WEIGHTS_DIR  = os.path.join(RESULTS_ROOT, "weights")

# for d in [RESULTS_ROOT, CURVES_DIR, MODELS_DIR, CM_DIR, WEIGHTS_DIR]:
#     os.makedirs(d, exist_ok=True)

# # ============================================================
# # SEED HELPER
# # ============================================================

# def set_seed(seed=None, deterministic=True):
#     if seed is None:
#         seed = SEED

#     os.environ["PYTHONHASHSEED"] = str(seed)
#     random.seed(seed)
#     np.random.seed(seed)
#     tf.random.set_seed(seed)

#     if deterministic:
#         os.environ["TF_DETERMINISTIC_OPS"] = "1"
#         try:
#             tf.config.experimental.enable_op_determinism()
#         except Exception:
#             pass

# set_seed(SEED)

# EXCLUDED_SUBJECTS = {"v56p"}

# # ============================================================
# # HIPERPARÁMETROS FIJOS DEL MODELO MULTI-STREAM
# # ============================================================

# ATTENTION_FIXED_MODEL_ARGS = {
#     "d_model": 64,
#     "num_layers": 2,
# }

# ATTENTION_FIXED_COMPILE_ARGS = {
#     "learning_rate": 1e-3,
# }

# # ============================================================
# # PREPARACIÓN DE STREAMS PARA X = (N, 19, 256)
# # ============================================================
# # Este bloque reemplaza prepare_streams_4s porque tu EEGDataset_ADHD_TF
# # genera ventanas de 2 segundos: 2 s * 128 Hz = 256 muestras.

# def prepare_streams_eeg(X, fs=128, n_win=10, n_bands=20):
#     """
#     X : ndarray (N, C, T)
#         Para tu caso: (N, 19, 256)

#     Retorna:
#         freq : (N, n_bands, 1)
#         temp : (N, n_win, 1)
#         spat : (N, C, 1)
#     """

#     X = np.asarray(X, dtype=np.float32)
#     N, C, T = X.shape

#     # --------------------------------------------------------
#     # 1) Spectral stream: potencia media en 20 bandas logarítmicas
#     # --------------------------------------------------------
#     log_bands = np.logspace(np.log10(1), np.log10(fs / 2), n_bands + 1)
#     bands = list(zip(log_bands[:-1], log_bands[1:]))

#     def band_power(signal):
#         nperseg = min(T, fs * 2)
#         f, Pxx = welch(signal, fs=fs, nperseg=nperseg)

#         values = []
#         for lo, hi in bands:
#             mask = (f >= lo) & (f < hi)
#             if np.any(mask):
#                 values.append(Pxx[mask].mean())
#             else:
#                 values.append(0.0)

#         return np.array(values, dtype=np.float32)

#     freq_feats = np.stack([
#         [band_power(ch) for ch in trial]
#         for trial in X
#     ]).astype(np.float32)                      # (N, C, n_bands)

#     freq = freq_feats.mean(axis=1)[..., None] # (N, n_bands, 1)

#     # --------------------------------------------------------
#     # 2) Temporal stream: 10 ventanas temporales
#     # --------------------------------------------------------
#     T_win = T // n_win
#     usable = n_win * T_win

#     x_trim = X[:, :, :usable]
#     windows = x_trim.reshape(N, C, n_win, T_win)

#     # Mismo espíritu del modelo original: media temporal simple
#     temp = windows.mean(axis=(1, 3))[..., None]  # (N, n_win, 1)

#     # --------------------------------------------------------
#     # 3) Spatial stream: RMS por canal
#     # --------------------------------------------------------
#     spat = np.sqrt((X ** 2).mean(axis=2))[..., None]  # (N, C, 1)

#     return (
#         freq.astype(np.float32),
#         temp.astype(np.float32),
#         spat.astype(np.float32),
#     )


# # ============================================================
# # NORMALIZACIÓN SIN FUGA DE INFORMACIÓN
# # ============================================================
# # Se ajusta SOLO con trainval y se aplica a test.

# def fit_stream_standardizer(train_streams, eps=1e-6):
#     stats = []

#     for Xs in train_streams:
#         mu = Xs.mean(axis=0, keepdims=True)
#         sd = Xs.std(axis=0, keepdims=True) + eps
#         stats.append((mu.astype(np.float32), sd.astype(np.float32)))

#     return stats


# def apply_stream_standardizer(streams, stats):
#     out = []

#     for Xs, (mu, sd) in zip(streams, stats):
#         out.append(((Xs - mu) / sd).astype(np.float32))

#     return tuple(out)


# def streams_to_input_dict(streams):
#     freq, temp, spat = streams

#     return {
#         "freq_input": freq,
#         "temp_input": temp,
#         "spat_input": spat,
#     }


# # ============================================================
# # DATASET BUILDER PARA EL MODELO MULTI-STREAM
# # ============================================================

# def make_multistream_ds_from_streams(
#     streams,
#     y,
#     groups=None,
#     training=False,
#     batch_size=64,
#     seed=123,
#     with_groups=False,
# ):
#     """
#     streams:
#         tuple(freq, temp, spat)

#     y:
#         etiquetas enteras (N,)

#     Si with_groups=False:
#         retorna (inputs_dict, y_onehot)

#     Si with_groups=True:
#         retorna (inputs_dict, y_int, group)
#         para evaluación por sujeto.
#     """

#     inputs = streams_to_input_dict(streams)

#     y_int = y.astype(np.int32)

#     if with_groups:
#         gg = groups.astype(str)
#         ds = tf.data.Dataset.from_tensor_slices((inputs, y_int, gg))
#     else:
#         y_oh = tf.keras.utils.to_categorical(y_int, num_classes=2).astype(np.float32)
#         ds = tf.data.Dataset.from_tensor_slices((inputs, y_oh))

#     if training:
#         ds = ds.shuffle(len(y_int), seed=seed, reshuffle_each_iteration=True)

#     ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

#     return ds


# # ============================================================
# # EXTRAER PROBABILIDAD POSITIVA DESDE SOFTMAX
# # ============================================================

# def extract_positive_prob_softmax(model_output):
#     out = model_output

#     if tf.is_tensor(out):
#         out = out.numpy()

#     out = np.asarray(out)

#     if out.ndim == 2 and out.shape[1] == 2:
#         return out[:, 1].astype(np.float32)

#     raise ValueError(f"Se esperaba salida softmax (B,2), pero llegó {out.shape}")


# # ============================================================
# # DATA
# # ============================================================

# root = "/kaggle/input/datasets/javierceron/tdha-data"
# adhd_dir = os.path.join(root, "ADHD")
# control_dir = os.path.join(root, "Control")

# dataset_tf = EEGDataset_ADHD_TF(
#     adhd_dir=adhd_dir,
#     control_dir=control_dir,
#     lowcut=0.5,
#     highcut=60.0,
#     notch=50.0,
#     window=2.0,
#     overlap=0.5,
#     default_fs=128
# )

# # Excluir sujetos problemáticos
# n_before = len(dataset_tf.samples)

# dataset_tf.samples = [
#     sample for sample in dataset_tf.samples
#     if os.path.splitext(sample[0])[0] not in EXCLUDED_SUBJECTS
# ]

# n_after = len(dataset_tf.samples)
# removed = n_before - n_after

# print(f"Excluded subjects: {sorted(EXCLUDED_SUBJECTS)}")
# print(f"Subjects removed: {removed}")
# print(f"Remaining subjects: {n_after}")

# if len(dataset_tf.samples) == 0:
#     raise ValueError("No subjects left after exclusion. Check EXCLUDED_SUBJECTS.")

# X, y, groups = build_epoch_arrays(dataset_tf)

# print("Epoch arrays:")
# print("  X:", X.shape, X.dtype)
# print("  y:", y.shape, "labels:", np.unique(y))
# print("  groups:", groups.shape, "n_subjects:", len(np.unique(groups.astype(str))))

# N_CH, N_T = int(X.shape[1]), int(X.shape[2])
# print("Model source shape (C,T):", (N_CH, N_T))

# if N_T != 256:
#     print(f"WARNING: Se esperaba T=256 por window=2.0 y fs=128, pero llegó T={N_T}")

# subj_names, subj_labels = build_subject_table_from_epochs(groups, y)

# print("Subject table:")
# print("  n_subjects:", len(subj_names))
# print("  Control:", int(np.sum(subj_labels == 0)), "| ADHD:", int(np.sum(subj_labels == 1)))


# # ============================================================
# # OUTER CV: SUBJECT-INDEPENDENT
# # ============================================================

# outer = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
# X_dummy = np.zeros((len(subj_names), 1), dtype=np.float32)

# final_results = []
# super_cm = np.zeros((2, 2), dtype=int)
# fold_histories = {}
# fold_cms = {}

# for fold_id, (trainval_subj_idx, test_subj_idx) in enumerate(
#     outer.split(X_dummy, subj_labels, groups=subj_names)
# ):
#     print(f"\n================ Fold {fold_id+1}/{N_FOLDS} ================")

#     inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED + fold_id)

#     inner_X_dummy = np.zeros((len(trainval_subj_idx), 1), dtype=np.float32)
#     inner_y = subj_labels[trainval_subj_idx]
#     inner_groups = subj_names[trainval_subj_idx]

#     tr_rel, val_rel = next(inner.split(inner_X_dummy, inner_y, groups=inner_groups))

#     train_subj_idx = trainval_subj_idx[tr_rel]
#     val_subj_idx   = trainval_subj_idx[val_rel]

#     tr_names = subj_names[train_subj_idx]
#     va_names = subj_names[val_subj_idx]
#     te_names = subj_names[test_subj_idx]

#     tr_labels = subj_labels[train_subj_idx]
#     va_labels = subj_labels[val_subj_idx]
#     te_labels = subj_labels[test_subj_idx]

#     if len(np.unique(tr_labels)) < 2:
#         raise ValueError(f"Fold {fold_id}: train set does not contain both classes.")
#     if len(np.unique(va_labels)) < 2:
#         raise ValueError(f"Fold {fold_id}: validation set does not contain both classes.")
#     if len(np.unique(te_labels)) < 2:
#         raise ValueError(f"Fold {fold_id}: test set does not contain both classes.")

#     tr_idx  = epoch_indices_from_subjects(groups, tr_names)
#     val_idx = epoch_indices_from_subjects(groups, va_names)
#     te_idx  = epoch_indices_from_subjects(groups, te_names)

#     trainval_names = np.concatenate([tr_names, va_names])
#     trainval_idx = epoch_indices_from_subjects(groups, trainval_names)

#     # ========================================================
#     # PREPARAR STREAMS POR FOLD
#     # ========================================================
#     # Importante:
#     # - Los streams se calculan por partición.
#     # - La normalización se ajusta SOLO en trainval.
#     # - Test nunca participa en el ajuste.

#     trainval_streams = prepare_streams_eeg(
#         X[trainval_idx],
#         fs=128,
#         n_win=10,
#         n_bands=20
#     )

#     test_streams = prepare_streams_eeg(
#         X[te_idx],
#         fs=128,
#         n_win=10,
#         n_bands=20
#     )

#     stream_stats = fit_stream_standardizer(trainval_streams)

#     trainval_streams = apply_stream_standardizer(trainval_streams, stream_stats)
#     test_streams = apply_stream_standardizer(test_streams, stream_stats)

#     trainval_ds_attention = make_multistream_ds_from_streams(
#         trainval_streams,
#         y[trainval_idx],
#         groups=None,
#         training=True,
#         batch_size=BATCH_SIZE,
#         seed=SEED,
#         with_groups=False,
#     )

#     test_ds_xyg = make_multistream_ds_from_streams(
#         test_streams,
#         y[te_idx],
#         groups=groups[te_idx],
#         training=False,
#         batch_size=BATCH_SIZE,
#         seed=SEED,
#         with_groups=True,
#     )

#     # ========================================================
#     # MODELO
#     # ========================================================

#     tf.keras.backend.clear_session()
#     set_seed(SEED)

#     model_args = dict(ATTENTION_FIXED_MODEL_ARGS)

#     model = build_eeg_attention_model(
#         freq_shape=(20, 1),
#         temp_shape=(10, 1),
#         spat_shape=(N_CH, 1),
#         d_model=model_args["d_model"],
#         num_layers=model_args["num_layers"],
#     )

#     opt = Adam(learning_rate=ATTENTION_FIXED_COMPILE_ARGS["learning_rate"])

#     model.compile(
#         optimizer=opt,
#         loss="categorical_crossentropy",
#         metrics=[
#             tf.keras.metrics.CategoricalAccuracy(name="acc"),
#             tf.keras.metrics.AUC(name="auc"),
#         ],
#     )

#     print("\nModel summary for fold", fold_id)
#     model.summary(line_length=150, expand_nested=True)

#     # ========================================================
#     # ENTRENAMIENTO FINAL EN TRAINVAL
#     # ========================================================

#     hist = model.fit(
#         trainval_ds_attention,
#         epochs=EPOCHS_FINAL,
#         verbose=1,
#     )

#     fold_histories[fold_id] = hist.history

#     model.save(os.path.join(MODELS_DIR, f"fold_{fold_id}.keras"))
#     model.save_weights(os.path.join(WEIGHTS_DIR, f"fold_{fold_id}.weights.h5"))

#     print(f"Fold {fold_id} model + weights saved. Trained {len(hist.history['loss'])} epochs.")

#     plot_fold_training_curve(
#         hist.history,
#         fold_id,
#         save_path=os.path.join(CURVES_DIR, f"fold_{fold_id}_curves.png"),
#         metrics=("loss", "acc", "auc"),
#     )

#     # ========================================================
#     # TEST EVALUATION A NIVEL DE SUJETO
#     # ========================================================

#     subject_preds = defaultdict(list)
#     subject_probs = defaultdict(list)
#     subject_true  = {}

#     for xb, yb, sb in test_ds_xyg:
#         raw_pred = model(xb, training=False)
#         prob = extract_positive_prob_softmax(raw_pred)
#         pred = (prob >= 0.5).astype(int)

#         y_np = yb.numpy().astype(int)
#         s_np = sb.numpy().astype(str)

#         for name, p, pr, yt in zip(s_np, pred, prob, y_np):
#             subject_preds[name].append(int(p))
#             subject_probs[name].append(float(pr))
#             subject_true[name] = int(yt)

#     y_true, y_pred, y_prob = [], [], []

#     for subj in sorted(subject_preds.keys()):
#         y_true.append(subject_true[subj])
#         y_pred.append(int(np.bincount(subject_preds[subj]).argmax()))
#         y_prob.append(float(np.mean(subject_probs[subj])))

#     acc  = accuracy_score(y_true, y_pred)
#     bacc = balanced_accuracy_score(y_true, y_pred)
#     f1   = f1_score(y_true, y_pred)

#     if len(np.unique(y_true)) < 2:
#         auc = np.nan
#     else:
#         auc = roc_auc_score(y_true, y_prob)

#     cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

#     super_cm += cm
#     fold_cms[fold_id] = cm

#     plot_fold_confusion_matrix(
#         cm,
#         fold_id,
#         save_path=os.path.join(CM_DIR, f"fold_{fold_id}_cm.png"),
#     )

#     final_results.append({
#         "fold": fold_id,
#         "accuracy": float(acc),
#         "balanced_acc": float(bacc),
#         "f1": float(f1),
#         "auc": float(auc) if not np.isnan(auc) else np.nan,
#         "model_args": dict(model_args),
#         "compile_args": dict(ATTENTION_FIXED_COMPILE_ARGS),
#         "stream_config": {
#             "fs": 128,
#             "n_win": 10,
#             "n_bands": 20,
#             "standardized_with_trainval_only": True,
#             "freq_shape": (20, 1),
#             "temp_shape": (10, 1),
#             "spat_shape": (N_CH, 1),
#         }
#     })

#     print(f"Fold {fold_id} | Acc={acc:.3f} | BAcc={bacc:.3f} | F1={f1:.3f} | AUC={auc:.3f}")
#     print("Confusion matrix:\n", cm)


# # ============================================================
# # REPORTES FINALES
# # ============================================================

# plot_all_training_curves(
#     fold_histories,
#     CURVES_DIR,
#     metrics=("loss", "acc", "auc"),
# )

# plot_all_confusion_matrices(
#     fold_cms,
#     super_cm,
#     CM_DIR
# )

# print("\nACCUMULATED CONFUSION MATRIX")
# print(super_cm)

# df_res = pd.DataFrame(final_results)

# print("\nFINAL RESULTS")
# print(df_res[["accuracy", "balanced_acc", "f1", "auc"]].agg(["mean", "std"]))

# df_res.to_csv(os.path.join(RESULTS_ROOT, "final_results.csv"), index=False)
# print("Saved:", os.path.join(RESULTS_ROOT, "final_results.csv"))

In [9]:
# ============================================================
# MULTI-STREAM TRANSFORMER FIJO: 5 folds fijos + 10 semillas
# 10 seeds x 5 folds = 50 corridas limpias
# ============================================================

MODEL_NAME = "eeg_attention_transformer"

SEED_FOLDS = 123
TRAINING_SEEDS = [123]  # 42,123, 456, 789, 2024, 7, 99, 314, 2718, 5000

N_FOLDS = 5
BATCH_SIZE = 64
EPOCHS_FINAL = 100

EXCLUDED_SUBJECTS = {"v56p"}

DATA_ROOT = "/kaggle/input/datasets/javierceron/tdha-data"

RESULTS_ROOT = "/kaggle/working/results_multistream_transformer_10seeds"

PARTITIONS_DIR = os.path.join(RESULTS_ROOT, "partitions")

for d in [RESULTS_ROOT, PARTITIONS_DIR]:
    os.makedirs(d, exist_ok=True)

PRINT_MODEL_SUMMARY_ONCE = True

# ============================================================
# SEED HELPER
# ============================================================

def set_all_seeds(seed, deterministic=True):
    tf.keras.backend.clear_session()
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    if deterministic:
        os.environ["TF_DETERMINISTIC_OPS"] = "1"
        try:
            tf.config.experimental.enable_op_determinism()
        except Exception:
            pass

# ============================================================
# HPs FIJOS MULTI-STREAM TRANSFORMER
# ============================================================

MULTISTREAM_FIXED_MODEL_ARGS = {
    "d_model": 64,
    "num_layers": 2,
}

MULTISTREAM_FIXED_COMPILE_ARGS = {
    "learning_rate": 1e-3,
}

STREAM_CONFIG = {
    "fs": 128,
    "n_win": 10,
    "n_bands": 20,
}

# ============================================================
# PREPARACIÓN DE STREAMS PARA X = (N, C, T)
# Compatible con EEGDataset_ADHD_TF: normalmente X = (N, 19, 256)
# ============================================================

def prepare_streams_eeg(X, fs=128, n_win=10, n_bands=20):
    """
    X : ndarray (N, C, T)

    Retorna:
        freq : (N, n_bands, 1)
        temp : (N, n_win, 1)
        spat : (N, C, 1)
    """

    X = np.asarray(X, dtype=np.float32)
    N, C, T = X.shape

    # --------------------------------------------------------
    # 1) Spectral stream: potencia media en bandas logarítmicas
    # --------------------------------------------------------
    log_bands = np.logspace(np.log10(1), np.log10(fs / 2), n_bands + 1)
    bands = list(zip(log_bands[:-1], log_bands[1:]))

    def band_power(signal):
        nperseg = min(T, fs * 2)
        f, Pxx = welch(signal, fs=fs, nperseg=nperseg)

        values = []
        for lo, hi in bands:
            mask = (f >= lo) & (f < hi)
            if np.any(mask):
                values.append(Pxx[mask].mean())
            else:
                values.append(0.0)

        return np.asarray(values, dtype=np.float32)

    freq_feats = np.stack([
        [band_power(ch) for ch in trial]
        for trial in X
    ]).astype(np.float32)                      # (N, C, n_bands)

    freq = freq_feats.mean(axis=1)[..., None] # (N, n_bands, 1)

    # --------------------------------------------------------
    # 2) Temporal stream: 10 ventanas temporales
    # --------------------------------------------------------
    T_win = T // n_win
    usable = n_win * T_win

    x_trim = X[:, :, :usable]
    windows = x_trim.reshape(N, C, n_win, T_win)

    # Misma idea del modelo original: media por ventana temporal
    temp = windows.mean(axis=(1, 3))[..., None]  # (N, n_win, 1)

    # --------------------------------------------------------
    # 3) Spatial stream: RMS por canal
    # --------------------------------------------------------
    spat = np.sqrt((X ** 2).mean(axis=2))[..., None]  # (N, C, 1)

    return (
        freq.astype(np.float32),
        temp.astype(np.float32),
        spat.astype(np.float32),
    )

# ============================================================
# UTILIDADES PARA STREAMS
# ============================================================

def slice_streams(streams, idxs):
    return tuple(S[idxs].astype(np.float32) for S in streams)


def fit_stream_standardizer(train_streams, eps=1e-6):
    """
    Ajusta normalización usando SOLO trainval.
    """
    stats = []

    for Xs in train_streams:
        mu = Xs.mean(axis=0, keepdims=True)
        sd = Xs.std(axis=0, keepdims=True) + eps
        stats.append((mu.astype(np.float32), sd.astype(np.float32)))

    return stats


def apply_stream_standardizer(streams, stats):
    out = []

    for Xs, (mu, sd) in zip(streams, stats):
        out.append(((Xs - mu) / sd).astype(np.float32))

    return tuple(out)


def streams_to_input_dict(streams):
    freq, temp, spat = streams

    return {
        "freq_input": freq,
        "temp_input": temp,
        "spat_input": spat,
    }

# ============================================================
# DATASET BUILDER MULTI-STREAM: softmax -> one-hot
# ============================================================

def make_multistream_ds_from_streams(
    streams,
    y,
    groups=None,
    training=False,
    batch_size=64,
    seed=123,
    with_groups=False,
):
    """
    Si with_groups=False:
        retorna (inputs_dict, y_onehot)

    Si with_groups=True:
        retorna (inputs_dict, y_int, group)
        para evaluación por sujeto.
    """

    inputs = streams_to_input_dict(streams)

    y_int = y.astype(np.int32)

    if with_groups:
        gg = groups.astype(str)
        ds = tf.data.Dataset.from_tensor_slices((inputs, y_int, gg))
    else:
        y_oh = tf.keras.utils.to_categorical(y_int, num_classes=2).astype(np.float32)
        ds = tf.data.Dataset.from_tensor_slices((inputs, y_oh))

    if training:
        ds = ds.shuffle(len(y_int), seed=seed, reshuffle_each_iteration=True)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return ds

# ============================================================
# EVALUACIÓN A NIVEL DE SUJETO PARA SOFTMAX
# ============================================================

def evaluate_subject_level_softmax(model, test_ds_xyg, threshold=0.5):
    subject_preds = defaultdict(list)
    subject_probs = defaultdict(list)
    subject_true = {}

    for xb, yb, sb in test_ds_xyg:
        raw_pred = model(xb, training=False).numpy()

        if raw_pred.ndim != 2 or raw_pred.shape[1] != 2:
            raise ValueError(f"Se esperaba salida softmax (B,2), pero llegó {raw_pred.shape}")

        prob = raw_pred[:, 1].astype(np.float32)
        pred = (prob >= threshold).astype(int)

        y_np = yb.numpy().astype(int)
        s_np = sb.numpy()

        for name, p, pr, yt in zip(s_np, pred, prob, y_np):
            name = name.decode("utf-8") if isinstance(name, bytes) else str(name)
            subject_preds[name].append(int(p))
            subject_probs[name].append(float(pr))
            subject_true[name] = int(yt)

    y_true, y_pred, y_prob = [], [], []

    for subj in sorted(subject_preds.keys()):
        y_true.append(subject_true[subj])
        y_pred.append(int(np.bincount(subject_preds[subj]).argmax()))
        y_prob.append(float(np.mean(subject_probs[subj])))

    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    if len(np.unique(y_true)) < 2:
        auc = np.nan
    else:
        auc = roc_auc_score(y_true, y_prob)

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    return {
        "accuracy": float(acc),
        "balanced_acc": float(bacc),
        "f1": float(f1),
        "auc": float(auc) if not np.isnan(auc) else np.nan,
        "cm": cm,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }

# ============================================================
# DATASET
# ============================================================

adhd_dir = os.path.join(DATA_ROOT, "ADHD")
control_dir = os.path.join(DATA_ROOT, "Control")

dataset_tf = EEGDataset_ADHD_TF(
    adhd_dir=adhd_dir,
    control_dir=control_dir,
    lowcut=0.5,
    highcut=60.0,
    notch=50.0,
    window=2.0,
    overlap=0.5,
    default_fs=128
)

n_before = len(dataset_tf.samples)

dataset_tf.samples = [
    sample for sample in dataset_tf.samples
    if os.path.splitext(sample[0])[0] not in EXCLUDED_SUBJECTS
]

n_after = len(dataset_tf.samples)

print(f"Excluded subjects: {sorted(EXCLUDED_SUBJECTS)}")
print(f"Subjects removed: {n_before - n_after}")
print(f"Remaining subjects: {n_after}")

if len(dataset_tf.samples) == 0:
    raise ValueError("No subjects left after exclusion.")

X, y, groups = build_epoch_arrays(dataset_tf)

N_CH, N_T = int(X.shape[1]), int(X.shape[2])

print("Epoch arrays:")
print("  X:", X.shape, X.dtype)
print("  y:", y.shape, "labels:", np.unique(y))
print("  groups:", groups.shape, "n_subjects:", len(np.unique(groups.astype(str))))
print("Source EEG shape (C,T):", (N_CH, N_T))

subj_names, subj_labels = build_subject_table_from_epochs(groups, y)

print("Subject table:")
print("  n_subjects:", len(subj_names))
print("  Control:", int(np.sum(subj_labels == 0)), "| ADHD:", int(np.sum(subj_labels == 1)))

# ============================================================
# PRECOMPUTAR STREAMS CRUDOS PARA TODO X
# ============================================================
# Esto no ajusta estadísticas globales ni usa etiquetas.
# La normalización sí se hace después, dentro de cada fold, usando solo trainval.

print("\nPrecomputing raw multi-stream features...")

ALL_STREAMS_RAW = prepare_streams_eeg(
    X,
    fs=STREAM_CONFIG["fs"],
    n_win=STREAM_CONFIG["n_win"],
    n_bands=STREAM_CONFIG["n_bands"],
)

print("Raw stream shapes:")
print("  freq:", ALL_STREAMS_RAW[0].shape)
print("  temp:", ALL_STREAMS_RAW[1].shape)
print("  spat:", ALL_STREAMS_RAW[2].shape)

# ============================================================
# PARTICIONES FIJAS
# ============================================================

outer = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED_FOLDS
)

X_dummy = np.zeros((len(subj_names), 1), dtype=np.float32)

FIXED_SPLITS = []
partitions_rows = []

for fold_id, (trainval_subj_idx, test_subj_idx) in enumerate(
    outer.split(X_dummy, subj_labels, groups=subj_names)
):
    print(f"\n================ FIXED FOLD {fold_id+1}/{N_FOLDS} ================")

    inner = StratifiedGroupKFold(
        n_splits=4,
        shuffle=True,
        random_state=SEED_FOLDS + fold_id
    )

    inner_X_dummy = np.zeros((len(trainval_subj_idx), 1), dtype=np.float32)
    inner_y = subj_labels[trainval_subj_idx]
    inner_groups = subj_names[trainval_subj_idx]

    tr_rel, val_rel = next(inner.split(inner_X_dummy, inner_y, groups=inner_groups))

    train_subj_idx = trainval_subj_idx[tr_rel]
    val_subj_idx = trainval_subj_idx[val_rel]

    tr_names = subj_names[train_subj_idx]
    va_names = subj_names[val_subj_idx]
    te_names = subj_names[test_subj_idx]

    tr_labels = subj_labels[train_subj_idx]
    va_labels = subj_labels[val_subj_idx]
    te_labels = subj_labels[test_subj_idx]

    if len(np.unique(tr_labels)) < 2:
        raise ValueError(f"Fold {fold_id}: train set does not contain both classes.")
    if len(np.unique(va_labels)) < 2:
        raise ValueError(f"Fold {fold_id}: validation set does not contain both classes.")
    if len(np.unique(te_labels)) < 2:
        raise ValueError(f"Fold {fold_id}: test set does not contain both classes.")

    tr_idx = epoch_indices_from_subjects(groups, tr_names)
    val_idx = epoch_indices_from_subjects(groups, va_names)
    te_idx = epoch_indices_from_subjects(groups, te_names)

    trainval_names = np.concatenate([tr_names, va_names])
    trainval_idx = epoch_indices_from_subjects(groups, trainval_names)

    FIXED_SPLITS.append({
        "fold_id": fold_id,
        "tr_names": tr_names,
        "va_names": va_names,
        "te_names": te_names,
        "tr_idx": tr_idx,
        "val_idx": val_idx,
        "te_idx": te_idx,
        "trainval_idx": trainval_idx,
    })

    partitions_rows.append({
        "fold": fold_id,
        "train_subjects": list(tr_names),
        "val_subjects": list(va_names),
        "test_subjects": list(te_names),
        "n_train_subjects": len(tr_names),
        "n_val_subjects": len(va_names),
        "n_test_subjects": len(te_names),
        "n_train_epochs": len(tr_idx),
        "n_val_epochs": len(val_idx),
        "n_test_epochs": len(te_idx),
    })

df_partitions = pd.DataFrame(partitions_rows)

df_partitions.to_csv(
    os.path.join(PARTITIONS_DIR, "fixed_partitions_summary.csv"),
    index=False
)

print("\nParticiones fijas guardadas en:")
print(os.path.join(PARTITIONS_DIR, "fixed_partitions_summary.csv"))

# ============================================================
# CORRER UNA SEED
# ============================================================

def run_multistream_one_seed(training_seed, save_artifacts=True):
    print(f"\n{'='*90}")
    print(f"RUN MULTI-STREAM TRANSFORMER | training_seed = {training_seed}")
    print(f"{'='*90}")

    seed_root = os.path.join(RESULTS_ROOT, f"seed_{training_seed}")
    curves_dir = os.path.join(seed_root, "training_curves")
    models_dir = os.path.join(seed_root, "models")
    cm_dir = os.path.join(seed_root, "cm")
    weights_dir = os.path.join(seed_root, "weights")

    for d in [seed_root, curves_dir, models_dir, cm_dir, weights_dir]:
        os.makedirs(d, exist_ok=True)

    fold_results = []
    fold_histories = {}
    fold_cms = {}
    super_cm = np.zeros((2, 2), dtype=int)

    global PRINT_MODEL_SUMMARY_ONCE

    for split in FIXED_SPLITS:
        fold_id = split["fold_id"]

        print(f"\n---------------- Fold {fold_id+1}/{N_FOLDS} | seed={training_seed} ----------------")

        fold_seed = int(training_seed * 100 + fold_id)

        trainval_idx = split["trainval_idx"]
        te_idx = split["te_idx"]

        # ====================================================
        # STREAMS DEL FOLD
        # ====================================================

        trainval_streams_raw = slice_streams(ALL_STREAMS_RAW, trainval_idx)
        test_streams_raw = slice_streams(ALL_STREAMS_RAW, te_idx)

        # Normalización ajustada SOLO con trainval
        stream_stats = fit_stream_standardizer(trainval_streams_raw)

        trainval_streams = apply_stream_standardizer(trainval_streams_raw, stream_stats)
        test_streams = apply_stream_standardizer(test_streams_raw, stream_stats)

        trainval_ds = make_multistream_ds_from_streams(
            trainval_streams,
            y[trainval_idx],
            groups=None,
            training=True,
            batch_size=BATCH_SIZE,
            seed=fold_seed,
            with_groups=False,
        )

        test_ds_xyg = make_multistream_ds_from_streams(
            test_streams,
            y[te_idx],
            groups=groups[te_idx],
            training=False,
            batch_size=BATCH_SIZE,
            seed=fold_seed,
            with_groups=True,
        )

        # ====================================================
        # MODELO
        # ====================================================

        set_all_seeds(fold_seed)

        model_args = dict(MULTISTREAM_FIXED_MODEL_ARGS)

        model = build_eeg_attention_model(
            freq_shape=(STREAM_CONFIG["n_bands"], 1),
            temp_shape=(STREAM_CONFIG["n_win"], 1),
            spat_shape=(N_CH, 1),
            d_model=model_args["d_model"],
            num_layers=model_args["num_layers"],
        )

        opt = Adam(learning_rate=MULTISTREAM_FIXED_COMPILE_ARGS["learning_rate"])

        model.compile(
            optimizer=opt,
            loss="categorical_crossentropy",
            metrics=[
                tf.keras.metrics.CategoricalAccuracy(name="acc"),
                tf.keras.metrics.AUC(name="auc"),
            ],
        )

        if PRINT_MODEL_SUMMARY_ONCE:
            print("\n================ MULTI-STREAM MODEL SUMMARY ================\n")
            model.summary(line_length=150, expand_nested=True)
            PRINT_MODEL_SUMMARY_ONCE = False

        # ====================================================
        # ENTRENAMIENTO
        # ====================================================

        hist = model.fit(
            trainval_ds,
            epochs=EPOCHS_FINAL,
            verbose=1,
        )

        fold_histories[fold_id] = hist.history

        if save_artifacts:
            # Guardado del modelo completo puede fallar si las capas custom
            # no tienen configuración serializable. Por eso se protege con try.
            try:
                model.save(os.path.join(models_dir, f"fold_{fold_id}.keras"))
            except Exception as e:
                print(f"WARNING: No se pudo guardar el modelo completo del fold {fold_id}: {e}")

            model.save_weights(os.path.join(weights_dir, f"fold_{fold_id}.weights.h5"))

            plot_fold_training_curve(
                hist.history,
                fold_id,
                save_path=os.path.join(curves_dir, f"fold_{fold_id}_curves.png"),
                metrics=("loss", "acc", "auc"),
            )

        # ====================================================
        # EVALUACIÓN SUBJECT-LEVEL
        # ====================================================

        fold_eval = evaluate_subject_level_softmax(
            model,
            test_ds_xyg,
            threshold=0.5
        )

        cm = fold_eval["cm"]
        super_cm += cm
        fold_cms[fold_id] = cm

        if save_artifacts:
            plot_fold_confusion_matrix(
                cm,
                fold_id,
                save_path=os.path.join(cm_dir, f"fold_{fold_id}_cm.png"),
            )

        row = {
            "seed": training_seed,
            "fold": fold_id,
            "accuracy": fold_eval["accuracy"],
            "balanced_acc": fold_eval["balanced_acc"],
            "f1": fold_eval["f1"],
            "auc": fold_eval["auc"],
            "model_name": MODEL_NAME,
            "model_args": dict(model_args),
            "compile_args": dict(MULTISTREAM_FIXED_COMPILE_ARGS),
            "stream_config": dict(STREAM_CONFIG),
            "freq_shape": (STREAM_CONFIG["n_bands"], 1),
            "temp_shape": (STREAM_CONFIG["n_win"], 1),
            "spat_shape": (N_CH, 1),
            "standardization": "fit_on_trainval_only",
        }

        fold_results.append(row)

        print(
            f"Fold {fold_id} | "
            f"Acc={row['accuracy']:.3f} | "
            f"BAcc={row['balanced_acc']:.3f} | "
            f"F1={row['f1']:.3f} | "
            f"AUC={row['auc']:.3f}"
        )

        print("Confusion matrix:\n", cm)

        del model
        gc.collect()
        tf.keras.backend.clear_session()

    if save_artifacts:
        plot_all_training_curves(
            fold_histories,
            curves_dir,
            metrics=("loss", "acc", "auc"),
        )

        plot_all_confusion_matrices(
            fold_cms,
            super_cm,
            cm_dir
        )

    df_seed = pd.DataFrame(fold_results)

    seed_summary = {
        "seed": training_seed,
        "mean_accuracy": float(df_seed["accuracy"].mean()),
        "std_accuracy": float(df_seed["accuracy"].std(ddof=1)),
        "mean_balanced_acc": float(df_seed["balanced_acc"].mean()),
        "std_balanced_acc": float(df_seed["balanced_acc"].std(ddof=1)),
        "mean_f1": float(df_seed["f1"].mean()),
        "std_f1": float(df_seed["f1"].std(ddof=1)),
        "mean_auc": float(df_seed["auc"].mean()),
        "std_auc": float(df_seed["auc"].std(ddof=1)),
    }

    df_seed.to_csv(os.path.join(seed_root, "fold_results.csv"), index=False)

    pd.DataFrame([seed_summary]).to_csv(
        os.path.join(seed_root, "seed_summary.csv"),
        index=False
    )

    print("\nResumen seed:")
    print(pd.DataFrame([seed_summary]).T)

    return df_seed, seed_summary

# ============================================================
# CORRER LAS 10 SEMILLAS
# ============================================================

all_fold_results = []
all_seed_summaries = []

for training_seed in TRAINING_SEEDS:
    df_seed, seed_summary = run_multistream_one_seed(
        training_seed=training_seed,
        save_artifacts=True
    )

    all_fold_results.append(df_seed)
    all_seed_summaries.append(seed_summary)

df_all_folds = pd.concat(all_fold_results, axis=0, ignore_index=True)
df_all_seeds = pd.DataFrame(all_seed_summaries)

df_all_folds.to_csv(
    os.path.join(RESULTS_ROOT, "all_fold_results.csv"),
    index=False
)

df_all_seeds.to_csv(
    os.path.join(RESULTS_ROOT, "all_seed_summaries.csv"),
    index=False
)

print("\nRESULTADOS POR SEED")
print(df_all_seeds)

print("\nRESUMEN GLOBAL")
print(
    df_all_seeds[
        ["mean_accuracy", "mean_balanced_acc", "mean_f1", "mean_auc"]
    ].agg(["mean", "std"])
)

print("\nArchivos principales guardados en:")
print(" -", os.path.join(RESULTS_ROOT, "all_fold_results.csv"))
print(" -", os.path.join(RESULTS_ROOT, "all_seed_summaries.csv"))
print(" -", os.path.join(PARTITIONS_DIR, "fixed_partitions_summary.csv"))

Excluded subjects: ['v56p']
Subjects removed: 1
Remaining subjects: 120
Epoch arrays:
  X: (16640, 19, 256) float32
  y: (16640,) labels: [0 1]
  groups: (16640,) n_subjects: 120
Source EEG shape (C,T): (19, 256)
Subject table:
  n_subjects: 120
  Control: 59 | ADHD: 61

Precomputing raw multi-stream features...
Raw stream shapes:
  freq: (16640, 20, 1)
  temp: (16640, 10, 1)
  spat: (16640, 19, 1)

================ FIXED FOLD 1/5 ================

================ FIXED FOLD 2/5 ================

================ FIXED FOLD 3/5 ================

================ FIXED FOLD 4/5 ================

================ FIXED FOLD 5/5 ================

Particiones fijas guardadas en:
/kaggle/working/results_multistream_transformer_10seeds/partitions/fixed_partitions_summary.csv

RUN MULTI-STREAM TRANSFORMER | training_seed = 123

---------------- Fold 1/5 | seed=123 ----------------

================ MULTI-STREAM MODEL SUMMARY ================



Model: "EEG_Attention_Transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━
┃ Layer (type)                               ┃ Output Shape                         ┃                 Param # ┃ Con
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━
│ freq_input (InputLayer)                    │ (None, 20, 1)                        │                       0 │ -  
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ temp_input (InputLayer)                    │ (None, 10, 1)                        │                       0 │ -  
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ spat_input (InputLayer)                    │ (None, 19, 1)                        │                       0 │ -  
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ dense (Dense)                              │ (None, 20, 64)                       │                     128 │ fre
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ dense_6 (Dense)                            │ (None, 10, 64)                       │                     128 │ tem
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ dense_12 (Dense)                           │ (None, 19, 64)                       │                     128 │ spa
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ positional_encoding (PositionalEncoding)   │ (None, 20, 64)                       │                       0 │ den
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ positional_encoding_1 (PositionalEncoding) │ (None, 10, 64)                       │                       0 │ den
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ positional_encoding_2 (PositionalEncoding) │ (None, 19, 64)                       │                       0 │ den
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ transformer_encoder_block                  │ (None, 20, 64)                       │                  83,200 │ pos
│ (TransformerEncoderBlock)                  │                                      │                         │    
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ transformer_encoder_block_2                │ (None, 10, 64)                       │                  83,200 │ pos
│ (TransformerEncoderBlock)                  │                                      │                         │    
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ transformer_encoder_block_4                │ (None, 19, 64)                       │                  83,200 │ pos
│ (TransformerEncoderBlock)                  │                                      │                         │    
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ transformer_encoder_block_1                │ (None, 20, 64)                       │                  83,200 │ tra
│ (TransformerEncoderBlock)                  │                                      │                         │    
├────────────────────────────────────────────┼──────────────────────────────────────┼─────────────────────────┼────
│ transformer_encoder_block_3                │ (None, 10, 64)                       │                  83,200 │ tra
│ (TransformerEncoderBlock)                  │          

 Total params: 574,082 (2.19 MB)

 Trainable params: 574,082 (2.19 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 26s 30ms/step - acc: 0.5736 - auc: 0.5886 - loss: 0.7160
Epoch 2/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - acc: 0.6825 - auc: 0.7259 - loss: 0.6128
Epoch 3/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - acc: 0.6921 - auc: 0.7541 - loss: 0.5906
Epoch 4/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - acc: 0.6947 - auc: 0.7586 - loss: 0.5828
Epoch 5/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - acc: 0.6992 - auc: 0.7680 - loss: 0.5760
Epoch 6/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 6s 30ms/step - acc: 0.7146 - auc: 0.7898 - loss: 0.5549
Epoch 7/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - acc: 0.7252 - auc: 0.7972 - loss: 0.5449
Epoch 8/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - acc: 0.7232 - auc: 0.7978 - loss: 0.5436
Epoch 9/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - acc: 0.7276 - auc: 0.8038 - loss: 0.5381
Epoch 10/100
204/204 ━━━━━━━━━━━━━━━━━━━━ 6s 29ms/step - acc: 0.7357 - auc: 0.8118 - loss: 0.5285
Epoch 11/100
204/204 ━━━━━━━